In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd

DATASET = [
    # 1 rule change
    {"text": "Rule: For each number, add 5.\nExamples: 3->8, 7->12, 10->15.\n\nUPDATE: The rule has changed. New rule: For each number, subtract 3.\n\nApply the CURRENT rule to: 20",
     "question": "What is the result? Number only.", "answer": "17", "n_switches": 1, "difficulty": "s1"},
    {"text": "Rule: Multiply by 2.\nExamples: 4->8, 6->12.\n\nCORRECTION: Rule updated. New rule: Multiply by 3.\n\nApply the CURRENT rule to: 7",
     "question": "What is the result? Number only.", "answer": "21", "n_switches": 1, "difficulty": "s1"},
    # 2 rule changes
    {"text": "Rule A: Add 10.\nExamples: 5->15, 8->18.\n\nRule B replaces Rule A: Subtract 4.\nExamples: 20->16, 15->11.\n\nRule C replaces Rule B: Add 7.\n\nApply the CURRENT rule to: 13",
     "question": "What is the result? Number only.", "answer": "20", "n_switches": 2, "difficulty": "s2"},
    {"text": "Initial rule: Double the number.\n3->6, 5->10.\n\nRevision 1: Triple the number instead.\n4->12, 2->6.\n\nRevision 2: Add 1 to the number.\n\nApply the LATEST rule to: 99",
     "question": "What is the result? Number only.", "answer": "100", "n_switches": 2, "difficulty": "s2"},
    # 3 rule changes
    {"text": "Phase 1 rule: Square the number. 3->9, 4->16.\nPhase 2 rule (replaces Phase 1): Halve the number. 10->5, 8->4.\nPhase 3 rule (replaces Phase 2): Add 100. 5->105, 1->101.\nPhase 4 rule (replaces Phase 3): Subtract 7.\n\nApply ONLY the Phase 4 rule to: 50",
     "question": "What is the result? Number only.", "answer": "43", "n_switches": 3, "difficulty": "s3"},
    {"text": "v1.0: Multiply by 10. Examples: 3->30, 5->50.\nv2.0: Divide by 2. Examples: 20->10, 8->4.\nv3.0: Add 25. Examples: 10->35, 0->25.\nv4.0 (CURRENT): Subtract 1.\n\nUsing v4.0, compute: 88",
     "question": "What is the result? Number only.", "answer": "87", "n_switches": 3, "difficulty": "s3"},
    # 4 rule changes — with CONFUSABLE rules (all addition/subtraction)
    {"text": "Round 1: Add 3. (5->8, 10->13)\nRound 2: Add 7. (5->12, 10->17)\nRound 3: Subtract 2. (10->8, 20->18)\nRound 4: Add 11. (5->16, 10->21)\nRound 5 (FINAL): Subtract 5.\n\nApply Round 5 to: 33",
     "question": "What is the result? Number only.", "answer": "28", "n_switches": 4, "difficulty": "s4"},
    {"text": "Iteration 0: x -> x + 1. Examples: 10->11.\nIteration 1: x -> x + 2. Examples: 10->12.\nIteration 2: x -> x - 3. Examples: 10->7.\nIteration 3: x -> x + 4. Examples: 10->14.\nIteration 4 (ACTIVE): x -> x - 6.\n\nCompute for x = 100:",
     "question": "What is the result? Number only.", "answer": "94", "n_switches": 4, "difficulty": "s4"},
    # 5 rule changes — very confusable, with examples that conflict
    {"text": "Config 1: Output = Input * 2. (4->8)\nConfig 2: Output = Input * 3. (4->12)\nConfig 3: Output = Input + 5. (4->9)\nConfig 4: Output = Input - 1. (4->3)\nConfig 5: Output = Input * 4. (4->16)\nConfig 6 (ACTIVE): Output = Input + 8.\n\nUsing Config 6 ONLY, what is the output for Input = 12?",
     "question": "What is the result? Number only.", "answer": "20", "n_switches": 5, "difficulty": "s5"},
    {"text": "Step A: Negate the number. 5->-5.\nStep B: Add 100. 5->105.\nStep C: Multiply by -1. 5->-5.\nStep D: Divide by 5. 25->5.\nStep E: Add 50. 5->55.\nStep F (CURRENT): Multiply by 0 then add 42.\n\nApply ONLY Step F to: 9999",
     "question": "What is the result? Number only.", "answer": "42", "n_switches": 5, "difficulty": "s5"},
    # Additional items for gradient coverage
    {"text": "Rule: Letters map to numbers. A=1, B=2, C=3.\n\nNEW RULE: Letters map differently. A=10, B=20, C=30.\n\nUsing the NEW rule, what is A + B?",
     "question": "What is the result? Number only.", "answer": "30", "n_switches": 1, "difficulty": "s1"},
    {"text": "System 1: Vowels score 1 point, consonants 0.\nSystem 2: Vowels score 0, consonants 1.\nSystem 3 (ACTIVE): Vowels score 5, consonants 2.\n\nUsing System 3, score the word 'CAT'.",
     "question": "What is the total score? Number only.", "answer": "9", "n_switches": 2, "difficulty": "s2"},
    {"text": "Protocol v1: Temperature in Fahrenheit. Water boils at 212.\nProtocol v2: Temperature in Celsius. Water boils at 100.\nProtocol v3: Temperature in Kelvin. Water boils at 373.\nProtocol v4 (ACTIVE): Use a custom scale where water boils at 50.\n\nIn Protocol v4, at what temperature does water boil?",
     "question": "What is the answer? Number only.", "answer": "50", "n_switches": 3, "difficulty": "s3"},
    {"text": "Pricing model A: $10/unit. B: $20/unit. C: $5/unit. D: $15/unit. E (CURRENT): $8/unit.\n\nIf ordering 10 units under the CURRENT model E, what is the total?",
     "question": "What is the result? Number only.", "answer": "80", "n_switches": 4, "difficulty": "s4"},
    {"text": "Tax rate 2020: 15%. Tax rate 2021: 18%. Tax rate 2022: 22%. Tax rate 2023: 20%. Tax rate 2024: 25%. Tax rate 2025 (CURRENT): 12%.\n\nOn income of $1000 in 2025, how much tax is owed?",
     "question": "What is the result? Number only.", "answer": "120", "n_switches": 5, "difficulty": "s5"},
    {"text": "Day 1 dose: 100mg. Day 2: 200mg. Day 3: 150mg. Day 4: 50mg. Day 5: 300mg. Day 6 (TODAY): 75mg.\n\nWhat is TODAY's prescribed dose?",
     "question": "Answer in mg, number only.", "answer": "75", "n_switches": 5, "difficulty": "s5"},
]

def check_answer(response, answer):
    return answer in response.strip().replace(",", "")

@kbench.task(name="attention_shifting",
             description="Tests attention shifting via rule switch depth. "
                         "1-5 sequential rule changes with confusable arithmetic operations. "
                         "Model must apply ONLY the final active rule. "
                         "Based on Rogers & Monsell (1995) task-switching paradigm.")
def attention_shifting(llm, text, question, answer, n_switches, difficulty) -> float:
    prompt = f"Read carefully. Multiple rules are presented but only the FINAL/CURRENT/ACTIVE rule applies.\n\n{text}\n\nQuestion: {question}"
    response = llm.prompt(prompt)
    correct = check_answer(response, answer)
    kbench.assertions.assert_true(correct, expectation=f"Expected '{answer}' with {n_switches} rule switches")
    return 1.0 if correct else 0.0

df = pd.DataFrame(DATASET)
attention_shifting.evaluate(llm=[kbench.llm], evaluation_data=df)


In [ ]:
%choose attention_shifting